# 14 — Medical Evaluation & Benchmarking

This notebook evaluates every checkpoint in the Medical SLM pipeline and produces a
summary comparison table.

**Checkpoints compared:**
| Checkpoint | Variable |
|---|---|
| Cosmopedia base (general LM) | `cfg.PRETRAIN_FINAL_CKPT` |
| DAPT (medical text) | `cfg.MED_DAPT_FINAL_CKPT` |
| SFT (instruction-tuned) | `cfg.MED_SFT_FINAL_CKPT` |

**Evaluation tasks:**
1. Medical perplexity — how well each model predicts held-out medical text
2. MCQ accuracy — likelihood scoring on USMLE-style questions
3. Calibration — does confidence correlate with correctness?
4. Qualitative generations on clinical prompts

## 2 — Evaluation Strategy

### Why perplexity alone is insufficient for Q&A tasks

**Perplexity** measures how well the model predicts held-out text at the token level.
A lower perplexity means the model assigns higher probability to the actual next token.
It is a good measure of language modelling quality but a poor proxy for task performance:

- A model can have low perplexity on medical text while answering MCQs randomly (50% accuracy)
  if it accurately predicts sentence continuations but cannot reason about answer choices
- Conversely, a model could have moderate perplexity but correctly identify answer letters

### Likelihood scoring for MCQ

For multiple-choice questions, we use **likelihood scoring**: feed each answer option to the
model as a completion of the question prompt, compute the log-probability, and pick the
option with the highest score.

This avoids sampling noise (we never generate — we score fixed strings) and directly measures
whether the model assigns more probability to the correct answer than to distractors.

This is the same approach used in MMLU, HellaSwag, and other standard LLM benchmarks.

## 1 — Setup

In [ ]:
import os, sys

if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.abspath(".."))

import config as cfg
cfg.make_dirs()
cfg.print_config()

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from tokenizer.preprocess import load_tokenizer
from model.gpt import GPT, GPTConfig
from evaluation.medical_metrics import (
    evaluate_mcq_accuracy,
    evaluate_medical_perplexity,
    run_usmle_benchmark,
)
import config as cfg

# Load the medical tokenizer (used for all medical checkpoints)
med_tok = load_tokenizer(cfg.MED_TOKENIZER_VOCAB, cfg.MED_TOKENIZER_MERGES)

MED_BLOCK_SIZE = 512

def load_model(ckpt_path):
    """Load a GPT model from a checkpoint path.

    Handles both raw state-dict files and checkpoints nested under a 'model' key.
    Always puts the model in eval mode after loading.

    Args:
        ckpt_path: path to the .pt checkpoint file

    Returns:
        GPT model in eval mode on device
    """
    config = GPTConfig(
        vocab_size=cfg.VOCAB_SIZE,
        block_size=MED_BLOCK_SIZE,
        n_layer=cfg.N_LAYERS,
        n_head=cfg.N_HEADS,
        n_embd=cfg.D_MODEL,
    )
    model = GPT(config).to(device)
    state = torch.load(ckpt_path, map_location=device)
    if isinstance(state, dict) and "model" in state:
        state = state["model"]
    model.load_state_dict(state)
    model.eval()
    print(f"Loaded: {ckpt_path}")
    return model

print("Utility functions ready.")

## 3 — Load Test Set

We hold out the last 1000 MedQA examples as the test set — these were never seen during
SFT training (training used only the first `cfg.MED_MEDQA_TRAIN_SIZE` examples).

**No data leakage:** We explicitly compute the slice indices from config constants so that
any change to `MED_MEDQA_TRAIN_SIZE` automatically updates the test split boundary.

In [ ]:
from datasets import load_dataset
import config as cfg

raw_medqa = load_dataset(cfg.MED_MEDQA_DATASET, split="train")

TEST_SIZE  = 1000
test_start = cfg.MED_MEDQA_TRAIN_SIZE
test_end   = min(test_start + TEST_SIZE, len(raw_medqa))

test_examples = list(raw_medqa.select(range(test_start, test_end)))

print(f"Total MedQA examples : {len(raw_medqa):,}")
print(f"Training range       : 0 – {cfg.MED_MEDQA_TRAIN_SIZE - 1:,}")
print(f"Test range           : {test_start:,} – {test_end - 1:,}")
print(f"Test set size        : {len(test_examples):,} examples")

## 4 — Medical Perplexity

We evaluate perplexity on the medical test split for both the DAPT and SFT models.

Note: the TinyStories model uses a *different tokenizer*, so its perplexity on the medical
token file is not directly comparable. We evaluate it separately using its own tokenizer
on the same raw text for reference.

**Expected results:**
- DAPT: ~30–50 perplexity (trained on this distribution)
- SFT: slightly higher (~35–55) — SFT may sacrifice some raw language modelling
  quality for instruction-following precision

In [ ]:
import math

# Load DAPT and SFT models sequentially to stay within VRAM budget
# WARNING: Loading both simultaneously would use ~2x VRAM — load one at a time
print("Evaluating DAPT model perplexity...")
dapt_model = load_model(cfg.MED_DAPT_FINAL_CKPT)
dapt_ppl = evaluate_medical_perplexity(dapt_model, device, split="test")
del dapt_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"  DAPT perplexity: {dapt_ppl:.2f}")

print("\nEvaluating SFT model perplexity...")
sft_model = load_model(cfg.MED_SFT_FINAL_CKPT)
sft_ppl = evaluate_medical_perplexity(sft_model, device, split="test")
del sft_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"  SFT perplexity : {sft_ppl:.2f}")

print()
print(f"{'Model':<20} {'Perplexity':>12}")
print("-" * 35)
print(f"{'DAPT':<20} {dapt_ppl:>12.2f}")
print(f"{'SFT':<20} {sft_ppl:>12.2f}")

## 5 — MCQ Accuracy: Step-by-Step Worked Example

Before running the full benchmark, we trace through one question manually.
This illustrates exactly how `evaluate_mcq_accuracy` works internally and why
it is a reliable evaluation method for MCQ tasks.

**Procedure:**
1. Format the question as a prompt (no answer appended)
2. For each option (A, B, C, D): append the option text and compute the average
   log-probability of the option tokens given the prompt
3. Select the option with the highest log-probability
4. Compare to the gold answer

In [ ]:
import torch
import torch.nn.functional as F
from loaders.medical import format_medqa_instruction
from tokenizer.preprocess import load_tokenizer
import config as cfg

sft_model = load_model(cfg.MED_SFT_FINAL_CKPT)
med_tok   = load_tokenizer(cfg.MED_TOKENIZER_VOCAB, cfg.MED_TOKENIZER_MERGES)

example = test_examples[0]   # first test example
fmt     = format_medqa_instruction(example)

print("=" * 70)
print("QUESTION PROMPT:")
print(fmt["prompt"])
print(f"\nGOLD ANSWER: {fmt['response']}")
print("=" * 70)

In [ ]:
@torch.no_grad()
def score_completion(model, prompt_ids, completion_ids, device):
    """Compute mean log-probability of completion tokens given prompt.

    This is the standard 'length-normalised likelihood' scoring used in MCQ evaluation.
    Length normalisation prevents the model from always favouring shorter answers.

    Args:
        model         : GPT model in eval mode
        prompt_ids    : list of int, encoded prompt tokens
        completion_ids: list of int, encoded completion (answer option) tokens
        device        : torch device

    Returns:
        float: mean log-probability per token (higher = more likely)
    """
    full_ids = prompt_ids + completion_ids
    x = torch.tensor(full_ids, dtype=torch.long, device=device).unsqueeze(0)

    logits = model(x[:, :-1])   # predict positions 1..T
    log_probs = F.log_softmax(logits, dim=-1)   # (1, T-1, vocab)

    # We only care about the positions corresponding to completion tokens
    start = len(prompt_ids) - 1   # index in the T-1 output sequence
    completion_log_probs = []
    for i, tok_id in enumerate(completion_ids):
        pos = start + i
        if pos < log_probs.shape[1]:
            completion_log_probs.append(log_probs[0, pos, tok_id].item())

    if not completion_log_probs:
        return float("-inf")
    return sum(completion_log_probs) / len(completion_log_probs)

# Encode the prompt
prompt_ids = med_tok.encode(fmt["prompt"]).ids

# Try common column names for answer options in MedQA
options = {}
for field in ["A", "B", "C", "D"]:
    for col_pattern in [field, f"option_{field}", f"option{field}", f"answer_{field}"]:
        if col_pattern in example:
            options[field] = example[col_pattern]
            break

# If options not found in expected fields, try to parse from the prompt
if not options:
    print("Could not find option fields. Raw example keys:", list(example.keys()))
    print("Showing available fields:")
    for k, v in example.items():
        print(f"  {k}: {str(v)[:100]}")
else:
    print(f"\nScoring {len(options)} options:")
    scores = {}
    for letter, option_text in options.items():
        option_ids = med_tok.encode(f" {letter}. {option_text}").ids
        score      = score_completion(sft_model, prompt_ids, option_ids, device)
        scores[letter] = score
        print(f"  Option {letter}: {option_text[:60]:60s}  log-prob = {score:.4f}")

    predicted = max(scores, key=scores.get)
    gold      = fmt["response"].strip()[0].upper() if fmt["response"] else "?"
    correct   = predicted == gold

    print(f"\nModel predicts: {predicted}")
    print(f"Gold answer   : {gold}")
    print(f"Correct       : {'YES ✓' if correct else 'NO ✗'}")
    print(f"\nScore gap (best vs second best): ", end="")
    sorted_scores = sorted(scores.values(), reverse=True)
    if len(sorted_scores) >= 2:
        print(f"{sorted_scores[0] - sorted_scores[1]:.4f} nats")

## 6 — Full Benchmark

We run `run_usmle_benchmark` from `evaluation/medical_metrics.py` for all three checkpoints.

Each checkpoint is loaded, evaluated, then unloaded before loading the next — this keeps
memory usage to one model at a time (safe on 16GB T4).

**Expected MCQ accuracy:**
- Cosmopedia base: ~25% (near random; no medical knowledge or instruction format)
- DAPT: ~30–35% (medical knowledge but no MCQ format training)
- SFT: ~45–55% (medical knowledge + MCQ format learned)

In [ ]:
from evaluation.medical_metrics import run_usmle_benchmark
import config as cfg

# Define checkpoints to evaluate
# The TinyStories model uses the same architecture but the TinyStories vocabulary —
# we use it as a baseline to quantify the value of DAPT + SFT
checkpoints = {
    "tinystories (base)": cfg.PRETRAIN_FINAL_CKPT,
    "dapt"             : cfg.MED_DAPT_FINAL_CKPT,
    "sft"              : cfg.MED_SFT_FINAL_CKPT,
}

def model_factory(ckpt_path):
    """Factory function: load a GPT model from a checkpoint path.

    run_usmle_benchmark calls this for each checkpoint, allowing it to load
    and unload models one at a time rather than holding all in memory.

    Args:
        ckpt_path: path to .pt checkpoint

    Returns:
        GPT model in eval mode on device
    """
    return load_model(ckpt_path)

results = run_usmle_benchmark(
    checkpoints=checkpoints,
    model_factory=model_factory,
    tokenizer=med_tok,
    device=device,
    examples=test_examples,
    max_examples=200,   # use 200 for speed; increase to 1000 for final paper-quality results
    verbose=True,
)

print("\n" + "=" * 50)
print("USMLE Benchmark Summary")
print("=" * 50)
print(f"{'Checkpoint':<25} {'Accuracy':>10}")
print("-" * 38)
for name, acc in results.items():
    print(f"{name:<25} {acc * 100:>9.1f}%")

## 7 — Calibration Analysis

A **calibrated** model is more confident when it is right and less confident when it is wrong.
We measure this by computing the score gap (best option log-prob minus second-best option
log-prob) and splitting by whether the model was correct.

If the model is well-calibrated:
- Average score gap on **correct** predictions should be higher
- Average score gap on **incorrect** predictions should be lower

Poor calibration (gaps similar regardless of correctness) is a common failure mode
in small models — it suggests the model is guessing rather than reasoning.

In [ ]:
# Calibration analysis on the SFT model — 20 examples
from loaders.medical import format_medqa_instruction
import config as cfg

sft_model = load_model(cfg.MED_SFT_FINAL_CKPT)

CALIB_N   = 20
gaps_correct   = []
gaps_incorrect = []

for ex in test_examples[:CALIB_N]:
    fmt        = format_medqa_instruction(ex)
    prompt_ids = med_tok.encode(fmt["prompt"]).ids
    gold       = fmt["response"].strip()[0].upper() if fmt["response"] else None

    # Collect option texts (try common field patterns)
    options = {}
    for letter in ["A", "B", "C", "D"]:
        for pat in [letter, f"option_{letter}", f"option{letter}"]:
            if pat in ex:
                options[letter] = ex[pat]
                break

    if not options or gold is None:
        continue

    scores = {}
    for letter, text in options.items():
        option_ids     = med_tok.encode(f" {letter}. {text}").ids
        scores[letter] = score_completion(sft_model, prompt_ids, option_ids, device)

    sorted_s  = sorted(scores.values(), reverse=True)
    gap       = sorted_s[0] - sorted_s[1] if len(sorted_s) >= 2 else 0.0
    predicted = max(scores, key=scores.get)

    if predicted == gold:
        gaps_correct.append(gap)
    else:
        gaps_incorrect.append(gap)

avg_gap_correct   = sum(gaps_correct)   / len(gaps_correct)   if gaps_correct   else float("nan")
avg_gap_incorrect = sum(gaps_incorrect) / len(gaps_incorrect) if gaps_incorrect else float("nan")

print(f"Calibration over {CALIB_N} examples")
print(f"  Correct predictions   : {len(gaps_correct):>3}  avg score gap = {avg_gap_correct:.4f}")
print(f"  Incorrect predictions : {len(gaps_incorrect):>3}  avg score gap = {avg_gap_incorrect:.4f}")
print()
if avg_gap_correct > avg_gap_incorrect:
    print("Model is CALIBRATED — higher confidence on correct answers.")
else:
    print("Model is NOT well calibrated — confidence does not correlate with correctness.")
    print("This is common for small models; DPO (notebook 15) can help.")

del sft_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Optional: scatter plot of score gap vs correctness
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4))
    if gaps_correct:
        ax.scatter(range(len(gaps_correct)), gaps_correct,
                   color="green", label=f"Correct (n={len(gaps_correct)})", alpha=0.7)
    if gaps_incorrect:
        ax.scatter(range(len(gaps_incorrect)), gaps_incorrect,
                   color="red", label=f"Incorrect (n={len(gaps_incorrect)})", alpha=0.7)
    ax.axhline(avg_gap_correct,   color="green", linestyle="--", alpha=0.5, label="Mean correct")
    ax.axhline(avg_gap_incorrect, color="red",   linestyle="--", alpha=0.5, label="Mean incorrect")
    ax.set_xlabel("Example index")
    ax.set_ylabel("Score gap (best − second-best log-prob)")
    ax.set_title("SFT Model Calibration — Score Gap by Correctness")
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available — skipping plot.")

## 8 — Sample Generations

We test the SFT model on 5 clinical prompts to qualitatively assess its medical reasoning
and communication style.

These are open-ended prompts rather than MCQs — they reveal whether the model can produce
coherent, clinically plausible explanations beyond just selecting an answer letter.

In [ ]:
from generation.sampler import generate, encode_prompt
import textwrap
import config as cfg

sft_model = load_model(cfg.MED_SFT_FINAL_CKPT)
med_tok   = load_tokenizer(cfg.MED_TOKENIZER_VOCAB, cfg.MED_TOKENIZER_MERGES)

CLINICAL_PROMPTS = [
    "A 65-year-old male presents with crushing chest pain radiating to the left arm. The most likely diagnosis is",
    "The mechanism by which ACE inhibitors lower blood pressure involves",
    "A patient with type 2 diabetes has an HbA1c of 9.5%. The first-line pharmacological treatment is",
    "Septic shock is distinguished from cardiogenic shock by",
    "The pathophysiology of asthma involves",
]

for prompt in CLINICAL_PROMPTS:
    x   = encode_prompt(prompt, med_tok, device)
    out = generate(
        sft_model, x, med_tok,
        block_size=MED_BLOCK_SIZE,
        max_new_tokens=130,
        temperature=0.7,
        top_k=40,
        top_p=0.9,
        repetition_penalty=1.1,
    )
    text = med_tok.decode(out[0].tolist())
    print(f"\nPROMPT: {prompt}")
    print(textwrap.fill(text, 80))
    print("─" * 80)

del sft_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Next Steps

Fill in the benchmark results table above, then:

- **Notebook 15** — DPO: leverage MedQA's labeled wrong options as rejected responses to
  improve the model's preference for correct medical answers
- **Notebook 16** — Upload all artifacts to HuggingFace Hub with a model card